In [1]:
pip install kafka-python

   ---------------------------------------- 0.0/326.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/326.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/326.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/326.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/326.1 kB ? eta -:--:--
   --- ------------------------------------ 30.7/326.1 kB 93.9 kB/s eta 0:00:04
   --- ------------------------------------ 30.7/326.1 kB 93.9 kB/s eta 0:00:04
   --- ------------------------------------ 30.7/326.1 kB 93.9 kB/s eta 0:00:04
   ----------- --------------------------- 92.2/326.1 kB 238.8 kB/s eta 0:00:01
   ----------- --------------------------- 92.2/326.1 kB 238.8 kB/s eta 0:00:01
   ----------- --------------------------- 92.2/326.1 kB 238.8 kB/s eta 0:00:01
   -------------- ----------------------- 122.9/326.1 kB 225.3 kB/s eta 0:00:01
   ---------------- --------------------- 143.4/326.1 kB 250.7 kB/s eta 0:00:01
 

In [2]:
import pandas as pd
from kafka import KafkaProducer
from time import sleep
from json import dumps
import json

In [4]:
producer = KafkaProducer(bootstrap_servers=['98.93.74.48:9092'], #change ip here
                         value_serializer=lambda x: 
                         dumps(x).encode('utf-8'))

In [5]:
producer.send('demo_test', value={'surnasdasdame':'parasdasdmar'})

In [6]:
df = pd.read_csv("data/indexProcessed.csv")

In [7]:
df.head()

,Index,Date,Open,High,Low,Close,Adj Close,Volume,CloseUSD
0,HSI,1986-12-31,2568.300049,2568.300049,2568.300049,2568.300049,2568.300049,0.0,333.879006
1,HSI,1987-01-02,2540.100098,2540.100098,2540.100098,2540.100098,2540.100098,0.0,330.213013
2,HSI,1987-01-05,2552.399902,2552.399902,2552.399902,2552.399902,2552.399902,0.0,331.811987
3,HSI,1987-01-06,2583.899902,2583.899902,2583.899902,2583.899902,2583.899902,0.0,335.906987
4,HSI,1987-01-07,2607.100098,2607.100098,2607.100098,2607.100098,2607.100098,0.0,338.923013


In [ ]:
while True:
    dict_stock = df.sample(1).to_dict(orient="records")[0]
    producer.send('demo_test', value=dict_stock)
    sleep(1)

In [ ]:
producer.flush() #clear data from kafka server

In [ ]:
import s3fs

# Fix: Remove region parameter or pass it through client_kwargs
s3 = s3fs.S3FileSystem(
    key="YOUR_AWS_ACCESS_KEY_ID",
    secret="YOUR_AWS_SECRET_ACCESS_KEY",
    client_kwargs={'region_name': 'us-east-1'}
)

print("S3 connection established successfully")

In [ ]:
import time
from botocore.exceptions import ClientError

# Reconnect S3 and consumer
s3 = s3fs.S3FileSystem(
    key="YOUR_AWS_ACCESS_KEY_ID",
    secret="YOUR_AWS_SECRET_ACCESS_KEY",
    client_kwargs={'region_name': 'us-east-1'}
)

consumer = KafkaConsumer(
    'demo_test',
     bootstrap_servers=['98.93.74.48:9092'],
    value_deserializer=lambda x: loads(x.decode('utf-8')))

print("Starting to consume messages with error handling...")
print("Press Ctrl+C to stop\n")

count = 0
error_count = 0
max_retries = 3

for i in consumer:
    retry_count = 0
    success = False
    
    while retry_count < max_retries and not success:
        try:
            with s3.open(f"s3://s3-kafka-data/stock_market_{count}.json", 'w') as file:
                json.dump(i.value, file)
            success = True
            
            # Print progress every 100 messages
            if count % 100 == 0:
                print(f"Processed {count} messages... (Errors: {error_count})")
            
            count += 1
            
        except Exception as e:
            retry_count += 1
            error_count += 1
            print(f"Error at message {count}, retry {retry_count}/{max_retries}: {str(e)[:100]}")
            
            if retry_count < max_retries:
                time.sleep(2)  # Wait before retry
            else:
                print(f"Failed to write message {count} after {max_retries} retries. Skipping...")
                count += 1
                break

print(f"\nCompleted! Total messages: {count}, Errors: {error_count}")